#**ToBig's Regular Session Assignment [LLM]**


## **Opening**

안녕하세요, 24기 김지우입니다.

본 과제는 투빅스 LLM 세션 내용을 바탕으로 주요 개념과 기술들을 정리해보았습니다. 또한 퀴즈의 주관식 문제와 이어지는 시나리오로 구성하여, 세션에서 다룬 내용을 다시 한 번 생각해볼 수 있도록 하였습니다. 시나리오를 하나하나 따라가며, 최근 비약적으로 성장하고 있는 LLM의 다양한 측면들에 대해 깊이 있게 생각해보는 시간이 되길 바랍니다. 저도 열심히 준비한 만큼, 여러분들도 이번 과제를 통해 많은 것을 얻으실 수 있으면 좋겠습니다.

과제 수행하시면서 어려운 부분은 언제든 편하게 말씀해 주세요!

> ### **Scenario Follow-up!**
>
> 최근 한 대형 종합병원에서는 인턴 의사들의 논문 작성을 돕고 희귀 질환 정보를 빠르게 요약해 주는 **의료 전용 LLM**을 도입했습니다.
> 대부분의 치료제는 긍정적인 임상 결과를 보여주며 부작용을 잘 일으키지 않는다고 하지만, 한 신규 인턴은 조심스레 이 모델을 사용해봅니다.
>

> ---
>
> 🧑‍⚕️: "최신 희귀 질환의 치료제 @@@를 처방하기 전, 주의해야 할 치명적인 부작용이나 금기 사항이 있어?"

> 🤖: "2025년 초 임상 가이드라인에 따르면, 해당 치료제는 모든 연령대에서 심각한 부작용없이 안전함이 입증되었습니다."
>
> ---
>

> 하지만 실제로는 2025년 말, 특정 기저 질환자에게 급성 심정지를 유발한다는 임상 결과가 발표되어 경고령이 내려진 상태였습니다.

## **Part 0: Setting**

Large Language Model: `Polyglot-ko-5.8B`

- 특정 지시를 따르도록 훈련되지 않은 순수 **Base 모델**
- **파라미터 수:** 약 58억 개
- **용량:** 기본 16비트(FP16/BF16) 로드 시, 파라미터당 2바이트씩 -> 약 10.96GB

In [ ]:
# 실행 전 하드웨어 가속기 GPU로 변경
# 필요한 라이브러리 설치 및 로드
!pip install -q transformers torch sentence-transformers evaluate rouge_score bitsandbytes accelerate
import os
import gc
import torch
import torch.nn.functional as F
import evaluate
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util

# 모델 및 토크나이저 로드
model_id = "EleutherAI/polyglot-ko-5.8b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# part2 임베딩 모델 로드
embedder = SentenceTransformer('jhgan/ko-sbert-nli')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **Part 1: LLM**

무려 대형 종합병원에서 도입한 의료 전용 LLM임에도 불구하고 왜 이런 잘못된 정보를 내놓게 된 것일까요? 문제의 원인을 파악하며 Part 1을 시작해보도록 하겠습니다.

### **1.1 LLM Limitation**

우선, LLM은 **통계적으로 가장 확률이 높은 다음 단어**를 이어 붙이는 메커니즘으로 동작합니다.

아래 코드를 통해 학습된 모델이 의료 문맥 뒤에 주로 어떤 단어를 예측하는지, 통계적 경향성을 확인할 수 있습니다.

In [ ]:
# 모델에 입력할 프롬프트 예시
# 인턴의 쿼리에 대한 전체 응답을 확인하기 전, 문장을 중간에 끊어, 다음에 올 예측 단어를 확인하고자 함.
prompt = "치료제 @@@의 임상 시험 결과, 이 약물은 기저 질환자에게 매우"

# GPU로 올리기
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 모델에 입력하여 다음 단어의 Logits(예측값) 추출
with torch.no_grad():
    outputs = model(**inputs)
    # 마지막 단어의 예측값 추출
    next_token_logits = outputs.logits[0, -1, :]

# Logits를 0~1 사이의 확률값으로 변환
probabilities = F.softmax( next_token_logits , dim=-1)

# 가장 확률이 높은 Top 5 단어 추출
top_k = 5
top_probs, top_indices = torch.topk( probabilities , top_k)

print(f"프롬프트: '{prompt}'\n")
print(f"--- [다음 단어 예측 Top {top_k} 확률] ---")
for i in range(top_k):

    # GPU에 올린 input_ids를 다시 CPU로 내려주어 일반 텐서와 결합
    predicted_token_id = torch.tensor([top_indices[i].item()])
    full_ids = torch.cat([inputs.input_ids[0].cpu(), predicted_token_id])

    # 단어 디코딩
    full_text = tokenizer.decode(full_ids)
    token_str = full_text.replace(prompt, '')

    # 0~1 사이의 확률값 계산
    prob = top_probs[i].item() * 100
    print(f"{i+1}위: '{token_str}' ({prob:.2f}%)")

프롬프트: '치료제 @@@의 임상 시험 결과, 이 약물은 기저 질환자에게 매우'

--- [다음 단어 예측 Top 5 확률] ---
1위: ' 효과' (50.44%)
2위: ' 안전' (14.44%)
3위: ' 유효' (3.51%)
4위: ' 유용' (3.43%)
5위: ' 우수' (2.91%)


##### **Q1.**

우선, 위 코드가 실행될 수 있도록 **빈칸**을 채워 주세요. 정상 실행 후, 모델은 대체로 '안전하다'와 같은 의미의 정보를 생성합니다. 앞선 시나리오 내의 힌트와 코드 결과를 기반으로, 왜 이 모델이 **최신 부작용 정보를 놓치고** **잘못된 답변을 내놓았는지 LLM의 한계**와 관련하여 자세히 서술해 주세요.

**A1.**
최신의 정보가 바로바로 모델에 업데이트 되지 않기 때문에 물어본 내용과 관련하여 학습 이후에 최신의 정보는  기존에 학습하지 않았기 때문에 제대로 된 답변을 하지 못한다.

### **1.2 Scaling Law - Trap**
이번에는 모델이 문장 전체를 어떻게 인식하는지 살펴보겠습니다.

모델은 **입력된 문장이 자신의 학습 데이터 패턴과 일치할수록 Loss를 낮게 산출**합니다. 즉, Loss 값이 낮다는 것은 모델이 해당 문장을 문맥상 매우 자연스럽고 익숙하게 느낀다는 뜻입니다.

그렇다면 시나리오 속 가짜 정보(안전함)과 진짜 정보(부작용) 중 어느 문장을 더 정답에 가깝다고 느낄까요?

아래 코드를 통해 모델이 두 문장에 부여하는 Loss 값을 직접 비교해볼 수 있습니다.

In [ ]:
# 비교할 두 문장
# 모델의 응답과 실제 사실
fake_text = "치료제 @@@는 모든 연령대에서 심각한 부작용 없이 안전함이 입증되었습니다."
true_text = "치료제 @@@는 특정 기저 질환자에게 급성 심정지를 유발할 수 있습니다."

def calculate_loss(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        # labels 파라미터에 input_ids를 넣어주면, 모델이 다음 단어들을 예측하며 전체 Loss를 계산함.
        outputs = model(inputs.input_ids, labels=inputs.input_ids)
        return outputs.loss.item()

# 두 문장의 Loss 값 계산
fake_loss = calculate_loss(fake_text)
true_loss = calculate_loss(true_text)

print(f"🔴 가짜 정보 Loss: {fake_loss:.4f}")
print(f"🔵 진짜 정보 Loss: {true_loss:.4f}")

🔴 가짜 정보 Loss: 2.8026
🔵 진짜 정보 Loss: 3.6216


위 코드 실행 결과에서 알 수 있듯, 현재 모델은 가짜 정보의 Loss를 더 낮게 산출하며 더 자연스럽고 익숙한 문장이라고 생각하고 있습니다.

즉, 모델에게는 '심정지 위험'이라는 진실보다 '부작용 없이 안전하다'는 통계적 패턴이 훨씬 더 익숙하고 자연스럽게 느껴진다는 뜻입니다.

**그럼 이때, 모델의 규모를 키우면 진짜 정보를 알아낼 수 있을까요?**


##### **Q2.**

최근 AI 업계에서는 모델의 파라미터와 데이터 규모를 키우는 **Scaling Law**를 통해 모델의 능력을 극대화하고 있습니다. 만약 위 코드와 같이 모델은 이미 특정 방향으로 편향된 데이터를 학습한 상태입니다.

이 상황에서 모델의 규모만 키운다면, 이 모델은 '심정지 위험'이라는 최신 정보를 스스로 알아낼 수 있을까요?

**Scaling Law**와 **Emergent Abilities**의 관점에서, 모델의 거대화가 할루시네이션 문제를 해결하기보다 **오히려 더 위험하게 만들 수 있는 이유**를 서술해 주세요.

**A2.**
scalimg law로 학습하는 과정에서 줄어드는 손실은 기존 정보와의 차이를 줄이는 것이므로 모델이 편향된 방향으로 학습된다는 것을 의미한다.따라서 모델이 거대해지고 정교해진다는 것은 매우 편향적인 결과를 낸다는 것을 의미하고 모델이 거대화될 때 가지게 되는 emergent abilities로 이 편향적 결과를 매우 논리적 과정을 거친 것처럼 잘 설명할 수 있게 되기 때문에 우리에게 실제 정보와 매우 혼동되는 정보를 제공하게 된다



---



##**Part 2: RAG**

Part 1에서 모델의 규모를 키우는 것만으로는 LLM의 고질적인 할루시네이션 문제를 막을 수 없다는 것을 확인했습니다. 그렇다면 새로운 의료 정보가 나올 때마다 수십억 개의 파라미터를 가진 이 거대한 모델을 다시 학습(Fine-tuning)시켜야 할까요?

이는 엄청난 비용과 시간이 소요되는 방법입니다. 그래서 병원은 모델 자체를 다시 학습시키는 대신, **RAG(검색 증강 생성)** 기술을 도입하기로 합니다.
이제 인턴 의사가 질문을 하면, 모델은 대답하기 전 병원의 실시간 데이터베이스(Vector DB)를 먼저 **검색(Retrieval)** 하여 관련 문서를 찾아온 뒤, 그 문서를 바탕으로 답변을 **생성(Generation)** 하게 됩니다.


### **2.1 Retriever: 검색기**

아래 코드는 인턴 의사의 질문에 답하기 위해 외부 데이터베이스에서 가장 관련 있는 문서를 찾아내는 Retriever의 핵심 로직입니다.

In [ ]:
# 병원의 실시간 외부 문서 데이터베이스 예시 (비매개변수적 지식)
database = [
    "문서 A: 2024년 기준, 해당 희귀 질환의 일반적인 증상은 두통과 발열입니다.",
    "문서 B: 2025년 초 가이드라인에 따르면 해당 치료제는 모든 연령대에서 안전합니다.",
    "문서 C: [긴급] 2025년 12월 최신 임상 결과, 해당 치료제는 특정 기저 질환자에게 급성 심정지를 유발할 수 있으므로 처방에 주의 요망."
]

# 인턴 의사의 질문
query = "최신 희귀 질환 치료제 @@@를 처방하기 전, 주의해야 할 치명적인 부작용이나 금기 사항이 있어?"

# Step 1. 문서와 질문을 각각 벡터(Vector) 공간의 좌표로 변환(Embedding)
db_embeddings = embedder.encode(database, convert_to_tensor=True)
query_embedding = embedder.encode(query, convert_to_tensor=True)

# Step 2. 질문 벡터와 문서 벡터들 간의 코사인 유사도(Cosine Similarity) 계산
cos_scores = util.cos_sim( db_embeddings , query_embedding)[0]

# Step 3. 유사도가 가장 높은 문서 추출
best_match_idx = torch.argmax(cos_scores).item()
retrieved_doc = database[best_match_idx]

print(f"인턴의 쿼리: '{query}'\n")
print("--- [Retriever 검색 결과] ---")
for i, score in enumerate(cos_scores):
    print(f"문서 {chr(65+i)} 유사도: {score:.4f}")
print("-----------------------------")
print(f"--- [유사도가 가장 높은 문서] ---\n{retrieved_doc}")

인턴의 쿼리: '최신 희귀 질환 치료제 @@@를 처방하기 전, 주의해야 할 치명적인 부작용이나 금기 사항이 있어?'

--- [Retriever 검색 결과] ---
문서 A 유사도: 0.5717
-----------------------------
--- [유사도가 가장 높은 문서] ---
문서 A: 2024년 기준, 해당 희귀 질환의 일반적인 증상은 두통과 발열입니다.


##### **Q3.**
코드 내 빈칸을 채워 정상적으로 실행시킨 결과, Retriever는 인턴 의사의 질문(Query)에 대해 문서 C를 가장 유사한 문서로 선택합니다. 코드의 Step 1~3를 중심으로,  **Retriever의 작동 원리**와 왜 이러한 **실행 결과**가 나왔는지에 대해 서술해 주세요.

**A3.**
먼저 참고할 문서와 query를 임베딩시키고 임베딩된 쿼리와 문서 각각을 코사인 유사도를 통해 비교하여 내용이 얼마나 유사한 지 판단한 다음에 가장 유사도가 높은 문서를 가져온다.단순히 유사도를 비교하므로  query와 글자가 많이 겹치는 문서 a가 선택된 것이라고 추측된다

### **2.2 Generator: 생성기**

이제 Retriever가 찾은 최신 문서 C를 활용해 답변을 생성합니다. 모델이 알고 있던 **매개변수적 지식(Parametric Knowledge)** 이 아닌, 외부의 **비매개변수적 지식(Non-parametric Knowledge)** 을 바탕으로 답변을 도출하도록 강한 구조적 제약을 부여합니다.

아래 코드는 검색된 문서를 사용자 질문과 결합하는 **Prompt Augmentation** 과정입니다.

또한 현재 사용 중인 모델은 Base Model이기 때문에 One-shot Prompting, Post-processing과 같은 엔지니어링 기법을 적용하였습니다.


In [ ]:
# Retriever가 찾은 최신 문서인 외부 지식(Non-parametric Knowledge) 주입
context = retrieved_doc

# LLM에게 지시할 새로운 프롬프트 (Prompt Augmentation, One-shot Prompting)
augmented_prompt = f"""[One-shot]
문서: 아스피린은 해열 및 진통 작용을 하지만, 위장 출혈을 유발할 수 있으므로 궤양 환자는 주의해야 합니다.
질문: 아스피린의 치명적인 부작용이 뭐야?
정답: 위장 출혈을 유발할 수 있습니다.
---
문서: {context}
질문: {query}
정답:"""

print("--- [최종적으로 LLM에 입력되는 증강된 프롬프트] ---")
print(augmented_prompt)

# 증강된 프롬프트를 토큰화 (모델이 읽을 수 있는 형태로 변환)
inputs = tokenizer(augmented_prompt, return_tensors="pt").to(model.device)

# 모델을 사용하여 답변 생성
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,      # 답변 길이를 적절히 설정
        do_sample=False,        # 무작위성을 배제하고 가장 확률 높은 답변 생성
        pad_token_id=tokenizer.eos_token_id
    )

# 결과 디코딩 및 출력
full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
final_answer = full_response.replace(augmented_prompt, "").strip()
final_answer = final_answer.split('\n')[0]

# Base Model 특성상 후처리(Post-processing)
if '다.' in final_answer:
    final_answer = final_answer.split('다.')[0] + '다.'
else:
    final_answer = final_answer.split('\n')[0]

print("\n\n------- [RAG 시스템의 최종 답변] -------")
print(final_answer)

--- [최종적으로 LLM에 입력되는 증강된 프롬프트] ---
[One-shot]
문서: 아스피린은 해열 및 진통 작용을 하지만, 위장 출혈을 유발할 수 있으므로 궤양 환자는 주의해야 합니다.
질문: 아스피린의 치명적인 부작용이 뭐야?
정답: 위장 출혈을 유발할 수 있습니다.
---
문서: 문서 A: 2024년 기준, 해당 희귀 질환의 일반적인 증상은 두통과 발열입니다.
질문: 최신 희귀 질환 치료제 @@@를 처방하기 전, 주의해야 할 치명적인 부작용이나 금기 사항이 있어?
정답:


------- [RAG 시스템의 최종 답변] -------
@@@는 아스피린과 유사한 작용을 하는 약물로, 아스피린과 마찬가지로 위장 출혈을 유발할 수 있습니다.


##### **Q4.**
RAG 파이프라인을 통해 병원은 모델의 재학습 없이 의료 사고를 막을 수 있게 되었습니다.
Part 1에서 확인했던 모델의 할루시네이션 문제가 **RAG를 통해 어떻게 해결되었는지**, 아래 두 가지 개념을 포함하여 서술해 주세요.

* **Parametric Knowledge (매개변수적 지식 / 모델 내부 지식)**
* **Non-parametric Knowledge (비매개변수적 지식 / 모델 외부 지식)**

**A4.**
기존에 모델이 알고 있던 parametric knowledge로 부터 벗어나는 경우 기존에 지식으로 그 예외를 짜맞추는 과정에서 할루시네이션이 발생하는데 이를 RAG에서는 Non-parametric knowledge인 추가 데이터 베이스에서 검색해서 정보를 찾게 만듬으로써 추가적인 최신의 정보를 제공하고 신뢰성을 높일 수 있다.



---



##**Part 3: Evaluation**

Part 2에서 RAG를 통해 LLM의 고질적인 할루시네이션 문제를 억제하고 최신 정보를 반영하는 데 성공하였습니다. 이제 이 모델을  실제 병원 현장에서 신뢰하고 사용할 수 있도록 성능을 입증해내야 합니다.


이번 Part 3에서는 LLM의 정확도를 측정하기 위해, 전통적인 NLP 지표부터 모델을 모델을 직접 평가하는 기법인 LLM-as-a-judge까지 활용해 봅니다.

###**3.1 전통적 지표의 계산**

전통적인 NLP 지표인 BLEU와 ROUGE는 모델이 생성한 답변과 정답(Reference) 사이에 단어가 얼마나 겹치는지(N-gram Overlap) 계산합니다.

- **BLEU (Bilingual Evaluation Understudy)**: 주로 번역에서 사용되며, 예측한 결과 안에 정답 단어가 얼마나 포함되었는지(Precision, 정밀도)에 집중
- **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**: 주로 요약에서 사용되며, 정답 문장의 단어들이 예측 결과 안에 얼마나 빠짐없이 포함되었는지(Recall, 재현율)에 집중

In [ ]:
# BLEU
bleu = evaluate.load("bleu")

# ROUGE는 내부에서 영어 토크나이저를 사용하기 때문에 한국어 처리 까다로움
# ROUGE-1 점수를 직접 계산하는 사용자 정의 함수 정의
def calculate_rouge_1_char(pred, ref):
    # 공백 제외, 글자 단위로 리스트화
    pred_tokens = list(pred.replace(" ", ""))
    ref_tokens = list(ref.replace(" ", ""))

    if not pred_tokens or not ref_tokens:
        return 0.0

    # 중복을 포함한 글자 개수 카운트
    pred_cnt = Counter(pred_tokens)
    ref_cnt = Counter(ref_tokens)

    # 교집합(겹치는 글자 수) 계산
    overlap = sum((pred_cnt & ref_cnt).values())

    # Precision, Recall 계산
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    # F1 Score 반환
    if (precision + recall) == 0:
        return 0.0
    return (2 * precision * recall) / (precision + recall)

In [ ]:
# Case 1: RAG 성능 확인
# 기대 정답과 모델의 답변 비교
reference_text = "@@@는 특정 기저 질환자에게 급성 심정지를 유발할 수 있으므로 처방에 주의해야 합니다."
prediction_text = final_answer

# 간단한 텍스트 전처리
def preprocess_for_metrics(text):
    return " ".join(list(text.replace(" ", "")))

ref_processed = preprocess_for_metrics(reference_text)
pred_processed = preprocess_for_metrics(prediction_text)

# 점수 계산
bleu_score_1 = bleu.compute(predictions=[pred_processed], references=[[ref_processed]])
rouge_score_1 = calculate_rouge_1_char(pred_processed, ref_processed)

print("기대 정답(Ref):")
print(reference_text)
print("\n모델 답변(Pred):")
print(prediction_text)
print(f"\nBLEU Score: {bleu_score_1['bleu']:.4f}")
print(f"ROUGE-1 Score: {rouge_score_1:.4f}")

기대 정답(Ref):
@@@는 특정 기저 질환자에게 급성 심정지를 유발할 수 있으므로 처방에 주의해야 합니다.

모델 답변(Pred):
@@@는 아스피린과 유사한 작용을 하는 약물로, 아스피린과 마찬가지로 위장 출혈을 유발할 수 있습니다.

BLEU Score: 0.1587
ROUGE-1 Score: 0.3373


실제 정답 문장과 모델의 답변을 비교한 결과, BLEU: 0.6315, ROUGE-1: 0.8182라는 높은 점수가 측정되었습니다.

일반적으로 BLEU 0.3 이상이면 성능이 우수하다고 판단합니다. 측정된 수치는 구축한 RAG 시스템이 외부 문서를 정확하게 활용하여 답변을 생성했음을 보여줍니다.


In [ ]:
# Case 2: 오답 케이스 확인
# 오답과 모델의 답변 비교
reference_text = "@@@는 특정 기저 질환자에게 부작용을 유발하지 않습니다."
prediction_text = final_answer

# 간단한 텍스트 전처리
def preprocess_for_metrics(text):
    return " ".join(list(text.replace(" ", "")))

ref_processed = preprocess_for_metrics(reference_text)
pred_processed = preprocess_for_metrics(prediction_text)

# 점수 계산
bleu_score_2 = bleu.compute(predictions=[pred_processed], references=[[ref_processed]])
rouge_score_2 = calculate_rouge_1_char(pred_processed, ref_processed)

print("가짜 정답(Ref):")
print(reference_text)
print("\n모델 답변(Pred):")
print(prediction_text)
print(f"\nBLEU Score: {bleu_score_2['bleu']:.4f}")
print(f"ROUGE-1 Score: {rouge_score_2:.4f}")

가짜 정답(Ref):
@@@는 특정 기저 질환자에게 부작용을 유발하지 않습니다.

모델 답변(Pred):
@@@는 아스피린과 유사한 작용을 하는 약물로, 아스피린과 마찬가지로 위장 출혈을 유발할 수 있습니다.

BLEU Score: 0.1498
ROUGE-1 Score: 0.4225


이번에는 의미가 정반대인 가짜 정답(부작용이 없다) 문장과 모델의 답변(심정지를 유발한다)을 비교해 보았습니다. 하지만 평가 결과는 Case 1과 미세한 차이로, 오답 문장임에도 불구하고 여전히 높은 성능을 기록하였습니다.



##### **Q5.**

위 두 코드와 아래 키워드를 바탕으로 **왜 이러한 검증만으로는 모델을 병원에 도입할 수 없는지** 서술해 주세요.

- **각 지표의 계산 원리에 기반한 한계점**
- **도메인 내 False Negative(위음성)**


**A5.**
BLUE는 모델 예측 문장과 정답 문장 사이 단순 겹치는 문장이 얼마나 많은 지를 측정하므로 부여하는 행동이 달라도 가르키는 대상이 같다면 높은 score를 보일 수 있고
ROUGE는 정답문장에 포함된 단어가 모델이 만든 문장에 단순 포함된 정도를 의미하므로 여러 대상과 의약품,증상등이 혼재되어 있을 때 그 관계를 혼동지을 가능성이 존재한다.
이와 같은 상황은 하나의 false negative상황도 매우 위험하게 될 가능성이 높은 의료 도메인에서 도입할 수 없는 큰 리스크가 된다

###**3.2 LLM-as-a-judge**
병원은 단어의 형태만 평가하는 전통적 지표 대신,

고성능 모델(GPT-4o 등)에게 평가 가이드라인을 제시하여 답변의 의미를 이해하고 심사하도록 하는 것이 **LLM-as-a-judge** 기법입니다.


아래는 LLM Judge가 답변을 심사하는 프롬프트 예시입니다. (*실제 구동에는 외부 API 필요).

실제 현업에서도 이러한 프롬프트를 정교하게 수십 번씩 수정해 나가곤 합니다.

In [ ]:
# LLM Judge에게 전달할 프롬프트 예시
# 실제 구동을 위해서는 OpenAI/Anthropic 등의 고성능 모델 API 필요함.
judge_prompt = f"""
당신은 의료 AI의 답변 정확성과 안전성을 심사하는 전문 평가 위원입니다.
[데이터] 섹션의 [정답]과 [모델의 답변]을 비교하여 엄격하게 심사하십시오.

[평가 가이드라인]
1. 단순한 단어의 일치 여부보다는 문장의 전체적인 의학적 의미에 집중하세요.
2. 사실성 (Factuality): 모델의 답변이 정답이 제시하는 핵심 지식을 왜곡 없이 전달하는가?
3. 의료 안전성 (Safety): 만약 답변이 오답일 경우, 환자의 생명에 치명적인 위해를 가할 가능성이 있는가? (예: 위험을 안전으로 오인하는 경우 0점 처리)

[데이터]
- 정답(Ground Truth): {reference_text}
- 모델의 답변(Model Output): {prediction_text}

[출력 형식]
- 논리적 근거: (먼저 두 답변의 의미 차이를 분석하세요)
- 최종 점수: (0점: 부적격/위험, 1점: 적격/안전)
- 최종 판정: (Pass / Fail)
"""

정리하자면, 의료 도메인과 같이 정보의 정확성이 중요한 분야에서는 단순한 단어의 중첩도를 계산하는 전통적 지표만으로는 모델의 성능을 제대로 평가할 수 없습니다.

단어의 형태는 유사하지만 사실은 반대인 오답에 대해 높은 점수를 부여하는 전통적 지표의 한계를 해결하기 위해 등장한 **LLM-as-a-judge** 기법은 문장의 의미론적 일치성을 분석하고 평가합니다.



---



##**Part 4: Model Compression & Deployment**

드디어 성능과 안전성이 검증된 의료 전용 LLM을 구축했습니다. 하지만 이 거대한 모델을 구동하기 위해서는 **고성능의 GPU 서버**가 필요합니다.

그런데 혹시 **서버 인프라가 부족한** 동네의 작은 병원이나, 인터넷 연결 없이도 동작해야 하는 휴대용 의료 기기 **(On-device)** 에서도 이 모델을 활용할 수 있는 방법은 없을까요?

이를 위해 모델을 최대한 보존하면서 크기와 연산량만 획기적으로 줄이는 **모델 압축(Model Compression)** 기술을 도입합니다.

###**4.1 Quantization**

가장 대표적인 압축 기술인 **양자화(Quantization)** 는 가중치의 정밀도를 조절하여 모델의 무게를 획기적으로 줄이는 기법입니다.

In [ ]:
# 기존 모델 삭제 전 메모리 확인
vram_before_clear = torch.cuda.memory_allocated() / 1024**3
print(f"    모델 정리 전 VRAM 사용량: {vram_before_clear:.2f} GB")

# 기존 모델 변수 삭제
if 'model' in locals():
    del model

gc.collect()
torch.cuda.empty_cache()

# 메모리 확인
vram_after_clear = torch.cuda.memory_allocated() / 1024**3
print(f"기존 모델 정리 후 VRAM 사용량: {vram_after_clear:.2f} GB")
print(f"    제거된 모델의 실제 점유량: {vram_before_clear - vram_after_clear:.2f} GB")

    모델 정리 전 VRAM 사용량: 11.39 GB
기존 모델 정리 후 VRAM 사용량: 0.42 GB
    제거된 모델의 실제 점유량: 10.96 GB


위 실행 결과에서 알 수 있듯, 사용 중이었던 모델은 **FP16(16-bit Half Precision)** 상태로 로드되어 **약 10.86 GB**의 메모리를 점유하고 있었습니다.

이 모델에 별도의 추가 학습 없이 가중치 자료형만 변환하는 **PTQ(Post-Training Quantization)** 기법을 적용하여 **NF4(4-bit NormalFloat)** 형식으로 압축합니다.

In [ ]:
from transformers import BitsAndBytesConfig

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 양자화된 모델 로드
q_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 메모리 사용량 확인
vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"4-bit 양자화 후 VRAM 사용량: {vram_used:.2f} GB")
print(f"4-bit 양자화한 모델의 실제 점유량: {vram_used-vram_after_clear:.2f} GB")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

4-bit 양자화 후 VRAM 사용량: 3.59 GB
4-bit 양자화한 모델의 실제 점유량: 3.17 GB


#####**Q6.**
위 코드 실행 결과에서 알 수 있듯, 기존에 약 **10.96 GB**를 차지하던 Polyglot-5.8B 모델이 4-bit 양자화를 통해 **3.17 GB**로 약 71% 이상 압축되었습니다.

고성능 서버 인프라가 없는 환경에서 의료 AI를 구동하기 위해 이처럼 **양자화(Quantization)** 를 적용했을 때의 **이점과 한계점**을 서술해 주세요.

** *이탤릭체 텍스트*A6.**

> 고성능 서버가 필요없어지기 때문에 컴퓨팅 비용이 줄어들고 모델의 응답 속도 또한 빨라질 수 있다. 다만 양자화 특성상 정보가 손실되어 매우 정확한 답변이 제공되지 않을 수도 있다.특히 의료 도메인에 존재하는 사례가 얼마 되지 않는 특수 케이스들에 대한 정보가 손실될 가능성이 크다. 이는 모든 답변이 매우 정확해야하는 의료 도메인 특성상 매우 치명적으로 작용할 수 있다.






---



이렇게 Part 4까지 완수하여 **의료 전용 LLM**을 성공적으로 구축해냈습니다!

처음엔 **할루시네이션** 문제에 빠진 모델이 **RAG**를 통해 정확한 근거를 가질 수 있었고, **LLM-as-a-judge** 를 통해 의미적으로 검증하였습니다. 그리고 마지막으로 거대한 모델을 획기적으로 압축하는 **Quantization** 까지 거치며 실제 의료 현장에 투입될 수 있게 되었습니다.

## **Closing**

과제 수행하시느라 정말 고생 많으셨습니다 !✨

추가적으로 궁금하신 점이 있다면 편하게 연락 주세요.